# Environment setup and Google Drive mount
In this section, we connect Google Drive to access the MVTec dataset and set up a secure destination to save the results. We also install any missing dependencies.

In [1]:
from google.colab import drive
import os
import shutil
import glob

# Mount Google Drive
drive.mount('/content/drive')

!git clone https://github.com/emanuelepietrocometti/sk-rd4ad.git
%cd /content/sk-rd4ad

!pip install --extra-index-url https://download.pytorch.org/whl/cu130 \
    torch>=2.1.0 torchvision>=0.16.0 numpy pandas scipy imageio matplotlib \
    opencv-python opencv-contrib-python Pillow scikit-image scikit-learn \
    fastprogress geomloss tqdm

Mounted at /content/drive
Cloning into 'sk-rd4ad'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 155 (delta 89), reused 63 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 86.30 KiB | 4.79 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/sk-rd4ad


# Configuration variables (CURRENT RUN)
Set the dataset class you want to train and the name of the results folder for this iteration. Update these values for each new run.

In [2]:
# === CONFIGURE YOUR RUN HERE ===
CLASS_NAME = "reda_dustValidationAndTrain"
RUN_NAME = "09072026_194825_reda_dustOnValidationAndTrain"

# Source and Destination Paths (Google Drive)
DRIVE_DATASET_PATH = f"/content/drive/MyDrive/Tesi/MVTec/{CLASS_NAME}"
DRIVE_RESULTS_PATH = f"/content/drive/MyDrive/Tesi/results/{RUN_NAME}"

# Local Paths (Fast Colab Storage)
LOCAL_DATASET_PATH = f"./mvtec/{CLASS_NAME}"
LOCAL_CHECKPOINT_PATH = "./checkpoints"
LOCAL_IMG_PATH = "./results_maps"

# Local dataset preparation
Deep Learning frameworks are significantly slower when reading images directly from Google Drive. This cell copies the dataset to Colab's local memory to maximize training speed.

In [3]:
print(f"Starting to copy the {CLASS_NAME} dataset from Drive to local Colab storage...")

# Clear the local folder if it already exists to avoid overlapping data
if os.path.exists(LOCAL_DATASET_PATH):
    shutil.rmtree(LOCAL_DATASET_PATH)

# Copy the files
shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
print("Dataset successfully copied!")

# Create local output directories
os.makedirs(LOCAL_CHECKPOINT_PATH, exist_ok=True)
os.makedirs(LOCAL_IMG_PATH, exist_ok=True)

Starting to copy the reda_dustValidationAndTrain dataset from Drive to local Colab storage...
Dataset successfully copied!


# Training phase
Execute the `main.py` script. This will use the default parameters (e.g., 200 epochs, res=3, learning rate 0.005) applied to the selected class.

In [4]:
print(f"--- STARTING TRAINING FOR CLASS: {CLASS_NAME} ---")

!python main.py \
  --class_ $CLASS_NAME \
  --data_path ./mvtec/ \
  --save_path $LOCAL_CHECKPOINT_PATH/ \
  --img_path $LOCAL_IMG_PATH/ \
  --seed 42 \
  --L2 0 \
  --res 1 \
  --rate 0.5 \
  --batch_size 4 \
  --layerloss 0 \
  --seg 1 \
  --vis 0 \
  --print_epoch 50 \
  --learning_rate 0.0006 \
  --cut 1

--- STARTING TRAINING FOR CLASS: reda_dustValidationAndTrain ---
--------args----------
epochs: 200
res: 1
learning_rate: 0.0006
batch_size: 4
seed: [42]
class_: reda_dustValidationAndTrain
seg: 1
print_epoch: 50
data_path: ./mvtec/
save_path: ./checkpoints/
print_canshu: 1
score_num: 1
print_loss: 1
img_path: ./results_maps/
vis: 0
cut: 1
layerloss: 0
rate: 0.5
print_max: 1
net: wide_res50
L2: 0
--------args----------

*************************
seed: 42
cuda
reda_dustValidationAndTrain
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth
100% 132M/132M [00:00<00:00, 210MB/s]
epoch [1/200], loss:0.9536
epoch [2/200], loss:0.6384
epoch [3/200], loss:0.5643
epoch [4/200], loss:0.4862
epoch [5/200], loss:0.4344
epoch [6/200], loss:0.4059
epoch [7/200], loss:0.3587
epoch [8/200], loss:0.3448
epoch [9/200], loss:0.3166
epoch [10/200], loss:0.2919
epoch [11/200], loss:0.2619
epoch [12/200], loss:0.2

# Evaluation and metrics report
Run the `eval.py` script to generate the final report with AUPRO, AP-loc, F1-Score, etc., and save the heatmaps categorized by TP, TN, FP, FN.
The script automatically detects the latest generated checkpoint.

In [5]:
print(f"--- STARTING EVALUATION FOR CLASS: {CLASS_NAME} ---")

# Automatically locate the most recently generated .pth file in the local directory
checkpoints = glob.glob(f"{LOCAL_CHECKPOINT_PATH}/*.pth")

if not checkpoints:
    print("ERROR: No checkpoints found. The training phase might have failed.")
else:
    # Select the newest file if multiple exist
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"Using checkpoint: {latest_checkpoint}")

    # Create a specific subfolder for validation maps
    EVAL_IMG_PATH = os.path.join(LOCAL_IMG_PATH, "eval_report")
    os.makedirs(EVAL_IMG_PATH, exist_ok=True)

    !python eval.py \
      --class_ $CLASS_NAME \
      --data_path ./mvtec/ \
      --checkpoint_path $latest_checkpoint \
      --img_path $EVAL_IMG_PATH/

--- STARTING EVALUATION FOR CLASS: reda_dustValidationAndTrain ---
Using checkpoint: ./checkpoints/wide_res50reda_dustValidationAndTrain10042sample_auc=0.91.pth
Using device: cuda
Loading checkpoint from: ./checkpoints/wide_res50reda_dustValidationAndTrain10042sample_auc=0.91.pth
Phase 1/2: Extracting anomaly scores and spatial maps...
Calculating pixel-level metrics (AUPRO, AP-loc, AUROC). This may take a moment...
 EVALUATION METRICS REPORT 
--- SAMPLE LEVEL (Image Classification) ---
Optimal Threshold: 0.2309
AUROC:             0.8929
Accuracy:          0.9097
F1-Score:          0.9444
Precision:         0.9297
Recall:            0.9597
Confusion Matrix (TN, FP | FN, TP):
[[ 44  18]
 [ 10 238]]

--- PIXEL LEVEL (Defect Localization) ---
AUPRO:             0.9235
AP-loc:            0.5101
Pixel AUROC:       0.9575

Phase 2/2: Saving visualizations categorized by Confusion Matrix...
Done! Evaluated images sorted in ./results_maps/eval_report/


# Hyperparameter Fine-Tuning (Optional)
Run this block only if you want to execute the hyperparameter grid search.

In [ ]:
print(f"--- STARTING HYPERPARAMETER TUNING FOR CLASS: {CLASS_NAME} ---")

!python hyperparameters_fine_tuning.py \
  --class_ $CLASS_NAME \
  --data_path ./mvtec/ \
  --save_path ./fine_tuning_checkpoints/

# Save results to Google Drive
Final synchronization. This step copies all trained weights (`.pth`) and visualizations (`.png`) securely to the preconfigured Google Drive directory.

In [6]:
print(f"Creating destination folder on Drive: {DRIVE_RESULTS_PATH}")
os.makedirs(DRIVE_RESULTS_PATH, exist_ok=True)

print("Transferring checkpoints...")
shutil.copytree(LOCAL_CHECKPOINT_PATH, os.path.join(DRIVE_RESULTS_PATH, "checkpoints"), dirs_exist_ok=True)

print("Transferring images and reports...")
shutil.copytree(LOCAL_IMG_PATH, os.path.join(DRIVE_RESULTS_PATH, "visualizations"), dirs_exist_ok=True)

# If fine-tuning was executed, save those results as well
if os.path.exists("./fine_tuning_checkpoints"):
    print("Transferring Fine-Tuning data...")
    shutil.copytree("./fine_tuning_checkpoints", os.path.join(DRIVE_RESULTS_PATH, "fine_tuning"), dirs_exist_ok=True)

print(f"✅ TRANSFER COMPLETE! All results are securely saved in: {DRIVE_RESULTS_PATH}")

Creating destination folder on Drive: /content/drive/MyDrive/Tesi/results/09072026_194825_reda_dustOnValidationAndTrain
Transferring checkpoints...
Transferring images and reports...
✅ TRANSFER COMPLETE! All results are securely saved in: /content/drive/MyDrive/Tesi/results/09072026_194825_reda_dustOnValidationAndTrain


In [10]:
!pip install onnx onnxruntime-gpu==1.22.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 14.1 MB/s eta 0:00:00
  Attempting uninstall: onnxruntime-gpu
    Found existing installation: onnxruntime-gpu 1.27.0
    Uninstalling onnxruntime-gpu-1.27.0:
      Successfully uninstalled onnxruntime-gpu-1.27.0


In [11]:
!python export_onnx_from_checkpoint.py /content/sk-rd4ad/checkpoints/wide_res50reda_dustValidationAndTrain10042sample_auc=0.91.pth /content/exports


--- Exporting SK-RD4AD (pure graph, weights: checkpoint:wide_res50reda_dustValidationAndTrain10042sample_auc=0.91.pth, res=3) -> /content/exports/wide_res50reda_dustValidationAndTrain10042sample_auc=0.91.onnx ---
/content/sk-rd4ad/export_onnx_from_checkpoint.py:180: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
[OK] fp32 export: /content/exports/wide_res50reda_dustValidationAndTrain10042sample_auc=0.91.onnx

--- PyTorch vs ONNX parity (atol=0.001, rtol=0.001) ---
  batch=1: map |Δ|max=3.19e-04  score |Δ|max=2.07e-04  OK
  batch=4: map |Δ|max=4.72e-04  score |Δ|max=1.37e-04  OK
[PASS] Numerical parity within tolerance confirmed.
